# Preprocesamiento de las capas de entrada

## 1. Configuración del entorno

Se importan las librerías fundamentales para el manejo de datos geoespaciales. `geopandas` gestiona las capas vectoriales, mientras que `rasterio` se utiliza para la manipulación eficiente de imágenes ráster. `leafmap` proporciona la interfaz interactiva para la selección de áreas. Se definen las rutas relativas para mantener la portabilidad del proyecto y se establece el sistema de referencia espacial EPSG:4326 (WGS 84), estándar para la interoperabilidad con servicios web y el paquete GeoAI.

In [ ]:
import os
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from rasterio.warp import transform_geom
from shapely.geometry import box, mapping
import leafmap.leafmap as leafmap
import matplotlib.pyplot as plt

# Configurar rutas relativas
RAW_DATA_PATH = '../data/raw'
PROCESSED_DATA_PATH = '../data/processed'
TRAIN_DATA_PATH = '../data/train'
CRS_PROJECT = 'EPSG:4326'  # WGS84 para compatibilidad con GeoAI y Web Maps

## 2. Definición del área de estudio y recorte vectorial

Se carga la capa vectorial correspondiente a la huella de la erupción proporcionada por el servicio de gestión de emergencias de Copernicus. Esta geometría define el alcance máximo del desastre y sirve como base para delimitar el área de interés (AOI). Se aplica un reproyección al CRS del proyecto si es necesario y se genera una envolvente rectangular (Bounding Box) con un margen de seguridad (*buffer*) de 0.005 grados para asegurar que el análisis cubra la totalidad de la zona afectada y su contexto inmediato.

In [ ]:
# Cargar la huella de la colada para definir el AOI
copernicus_path = f'{RAW_DATA_PATH}/area_colada.json'
gdf_copernicus = gpd.read_file(copernicus_path)

# Obtener Bounding Box
if gdf_copernicus.crs != CRS_PROJECT:
    gdf_copernicus = gdf_copernicus.to_crs(CRS_PROJECT)

aoi_geometry = gdf_copernicus.unary_union.envelope.buffer(0.005)
gdf_aoi = gpd.GeoDataFrame(geometry=[aoi_geometry], crs=CRS_PROJECT)

print(f'AOI definido. Extensión: {gdf_aoi.total_bounds}')
gdf_aoi.to_file(f'{PROCESSED_DATA_PATH}/vector/aoi_boundary.geojson', driver='GeoJSON')

AOI definido. Extensión: [-17.939693  28.594175 -17.858332  28.63732 ]


En esta etapa, se procesan las capas vectoriales de referencia: el catastro y los límites municipales. Se incluyen también los edificios afectados según Copernicus para el análisis final. Se itera sobre cada archivo, asegurando primero que su sistema de coordenadas coincida con el del proyecto. A continuación, se aplica una operación de recorte geométrico (`clip`) utilizando el AOI definido anteriormente. Este paso permite reducir el volumen de datos a procesar, eliminando la información geográfica irrelevante fuera de la zona de emergencia.

In [ ]:
# Recortar capas vectoriales
vector_layers = {
    'catastro': f'{RAW_DATA_PATH}/catastro_lapalma.geojson',
    'municipios': f'{RAW_DATA_PATH}/limites_municipales.geojson',
    'detecciones': f'{RAW_DATA_PATH}/detecciones_copernicus.json'
}

gdf_catastro_aoi = None

for name, path in vector_layers.items():
    print(f'Procesando capa: {name}...')
    gdf = gpd.read_file(path)
    if gdf.crs != CRS_PROJECT:
        gdf = gdf.to_crs(CRS_PROJECT)
        
    gdf_clipped = gpd.clip(gdf, gdf_aoi)
    
    if name == 'catastro':
        gdf_catastro_aoi = gdf_clipped
    
    output_file = f'{PROCESSED_DATA_PATH}/vector/{name}_clipped.geojson'
    gdf_clipped.to_file(output_file, driver='GeoJSON')
    print(f'  -> Guardado en {output_file} ({len(gdf_clipped)} geometrías)')

Procesando capa: catastro...
  -> Guardado en ../data/processed/vector/catastro_clipped.geojson (3661 geometrías)
Procesando capa: municipios...
  -> Guardado en ../data/processed/vector/municipios_clipped.geojson (3 geometrías)
Procesando capa: detecciones...
  -> Guardado en ../data/processed/vector/detecciones_clipped.geojson (4625 geometrías)


## 3. Procesamiento de imágenes ráster

Se define una función `clip_raster` para el procesamiento de las ortoimágenes. Esta función recorta la imagen a la geometría dada, además de optimizar el archivo de salida convirtiéndolo en un *Cloud Optimized GeoTIFF (COG)*. Se aplica compresión LZW para reducir el tamaño sin pérdida de calidad y se genera una estructura interna de teselas (`tiled=True`). Adicionalmente, se construyen pirámides de resolución (`overviews`) para permitir una visualización fluida a diferentes escalas en el mapa interactivo posterior.

In [ ]:
from rasterio.enums import Resampling
def clip_raster(input_path, output_path, aoi_geom, aoi_crs):
    with rasterio.open(input_path) as src:
        if src.crs != aoi_crs:
            geom_mapping = mapping(aoi_geom)
            transformed_geom = transform_geom(aoi_crs, src.crs, geom_mapping)
            shapes = [transformed_geom]
        else:
            shapes = [mapping(aoi_geom)]
        # Recortar capas ráster
        out_image, out_transform = mask(src, shapes, crop=True)
        out_meta = src.meta.copy()

        out_meta.update({
            'driver': 'GTiff',
            'height': out_image.shape[1],
            'width': out_image.shape[2],
            'transform': out_transform,
            'crs': src.crs,
            'compress': 'lzw',
            'tiled': True,
            'blockxsize': 512,
            'blockysize': 512,
            'predictor': 2
        })
        with rasterio.open(output_path, 'w', **out_meta) as dest:
            dest.write(out_image)
            factors = [2, 4, 8, 16] # Añadir mosáicos para una visualización eficiente
            dest.build_overviews(factors, Resampling.average)
            dest.update_tags(ns='rio_overview', resampling='average')
            
raster_files = {
    'orto_2021': f'{RAW_DATA_PATH}/ortofoto_2021.tif',
    'orto_2024': f'{RAW_DATA_PATH}/ortofoto_2024.tif'
}

aoi_poly = gdf_aoi.geometry.iloc[0]
aoi_crs = gdf_aoi.crs

for name, path in raster_files.items():
    print(f'Procesando imagen: {name}...')
    output_file = f'{PROCESSED_DATA_PATH}/raster/{name}_clipped.tif'
    clip_raster(path, output_file, aoi_poly, aoi_crs)
    print(f'  -> Imagen recortada y optimizada guardada en {output_file}')

Procesando imagen: orto_2021...
  -> Imagen recortada y optimizada guardada en ../data/processed/raster/orto_2021_clipped.tif
Procesando imagen: orto_2024...
  -> Imagen recortada y optimizada guardada en ../data/processed/raster/orto_2024_clipped.tif


## 4. Selección interactiva de zonas de entrenamiento

Se despliega un mapa interactivo centrado en la zona cero cargando la ortofoto post-erupción (2024). El usuario puede dibujar un rectángulo para definir una zona específica para el entrenamiento del modelo. Si no se realiza ninguna selección manual, el sistema aplica automáticamente unas coordenadas predeterminadas que abarcan un núcleo urbano elegido arbitrariamente. El script procede a recortar tanto la imagen ráster como la capa vectorial de catastro correspondiente a esa selección, guardando los pares resultantes en la estructura de directorios preparada para el entrenamiento.

In [ ]:
# Dibujar área de entrenamiento 2024 (opcional)
m1 = leafmap.Map(center=[28.607, -17.881], zoom=15)
m1.add_raster(f'{PROCESSED_DATA_PATH}/raster/orto_2024_clipped.tif', layer_name='Ortofoto 2024')
m1

Map(center=[28.615747, -17.899026999999997], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_…

In [ ]:
# Sin dibujo, utilizar zona por defecto
bbox_train_2024 = m1.user_roi_bounds()
if bbox_train_2024 is None:
    bbox_train_2024 = (-17.8855691821927358, 28.6047402071258787, -17.8773047141331602, 28.6098019752305497)

# Recorte ráster
clip_raster(f'{PROCESSED_DATA_PATH}/raster/orto_2024_clipped.tif', f'{TRAIN_DATA_PATH}/images/train_2024.tif', box(*bbox_train_2024), CRS_PROJECT)

# Recorte vectorial (máscara)
mask_2024 = gpd.clip(gdf_catastro_aoi, box(*bbox_train_2024))
mask_2024.to_file(f'{TRAIN_DATA_PATH}/masks/train_2024.geojson', driver='GeoJSON')

print(f"Área de entrenamiento (2024) guardada: {bbox_train_2024}")
print(f"  -> Imagen: {TRAIN_DATA_PATH}/images/train_2024.tif")
print(f"  -> Máscara: {TRAIN_DATA_PATH}/masks/train_2024.geojson ({len(mask_2024)} geometrías)")

Área Train 2024 guardada: (-17.885569182192736, 28.60474020712588, -17.87730471413316, 28.60980197523055)
  -> Imagen: ../data/train/images/train_2024.tif
  -> Máscara: ../data/train/masks/train_2024.geojson (112 geometrías)


De manera análoga, se selecciona y procesa el área de entrenamiento sobre la imagen previa a la erupción (2021). Se consideran muestras de ambas fechas para que el modelo aprenda a generalizar la apariencia de los edificios bajo diferentes condiciones de iluminación y contextos temporales.

In [ ]:
# Dibujar área de entrenamiento 2021 (opcional)
m2 = leafmap.Map(center=[28.627, -17.903], zoom=15)
m2.add_raster(f'{PROCESSED_DATA_PATH}/raster/orto_2021_clipped.tif', layer_name='Ortofoto 2021')
m2

Map(center=[28.615747, -17.899026999999997], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_…

In [ ]:
# Sin dibujo, utilizar zona por defecto
bbox_train_2021 = m2.user_roi_bounds()
if bbox_train_2021 is None:
    bbox_train_2021 = (-17.9074852686126960, 28.6250624248900287, -17.8992208005318290, 28.6301241930252388)

# Recorte ráster
clip_raster(f'{PROCESSED_DATA_PATH}/raster/orto_2021_clipped.tif', f'{TRAIN_DATA_PATH}/images/train_2021.tif', box(*bbox_train_2021), CRS_PROJECT)

# Recorte vectorial (máscara)
mask_2021 = gpd.clip(gdf_catastro_aoi, box(*bbox_train_2021))
mask_2021.to_file(f'{TRAIN_DATA_PATH}/masks/train_2021.geojson', driver='GeoJSON')

print(f"Área de entrenamiento (2021) guardada: {bbox_train_2021}")
print(f"  -> Imagen: {TRAIN_DATA_PATH}/images/train_2021.tif")
print(f"  -> Máscara: {TRAIN_DATA_PATH}/masks/train_2021.geojson ({len(mask_2021)} geometrías)")

Área Train 2021 guardada: (-17.907485268612696, 28.62506242489003, -17.89922080053183, 28.63012419302524)
  -> Imagen: ../data/train/images/train_2021.tif
  -> Máscara: ../data/train/masks/train_2021.geojson (150 geometrías)


Finalmente, se define un área de prueba independiente, que no será utilizada durante el entrenamiento de la red neuronal. Esta zona servirá exclusivamente para la validación del modelo (cuaderno `02_creacion_modelo.ipynb`), permitiendo evaluar su capacidad para detectar edificios en zonas que "desconocidas", garantizando así una medida objetiva de su rendimiento.

In [ ]:
# Dibujar área de validación (opcional)
m3 = leafmap.Map(center=[28.616, -17.898], zoom=15)
m3.add_raster(f'{PROCESSED_DATA_PATH}/raster/orto_2021_clipped.tif', layer_name='Ortofoto 2021')
m3

Map(center=[28.615747, -17.899026999999997], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_…

In [ ]:
# Sin dibujo, utilizar zona por defecto
bbox_test = m3.user_roi_bounds()
if bbox_test is None:
    bbox_test = (-17.9026522345241403, 28.6136674796345432, -17.8943877664479665, 28.6187292477532758)

# Recorte ráster
clip_raster(f'{PROCESSED_DATA_PATH}/raster/orto_2021_clipped.tif', f'{TEST_DATA_PATH}/images/test_2021.tif', box(*bbox_test), CRS_PROJECT)

# Recorte vectorial (máscara)
mask_test = gpd.clip(gdf_catastro_aoi, box(*bbox_test))
mask_test.to_file(f'{TEST_DATA_PATH}/masks/test_2021.geojson', driver='GeoJSON')

print(f"Área de validación guardada: {bbox_test}")
print(f"  -> Imagen: {TEST_DATA_PATH}/images/test_2021.tif")
print(f"  -> Máscara: {TEST_DATA_PATH}/masks/test_2021.geojson ({len(mask_test)} geometrías)")

Área Test guardada: (-17.90265223452414, 28.613667479634543, -17.894387766447966, 28.618729247753276)
  -> Imagen: ../data/test/images/test_2021.tif
  -> Máscara: ../data/test/masks/test_2021.geojson (117 geometrías)
